# DINO Feature Extraction — STL-10

Extracts DINO ViT-S/8 embeddings for the STL-10 train and test splits.

Output files (saved to Google Drive):
- `stl10_dino_train_embeddings.pt` — dict with keys `features` (5000, 384) and `labels` (5000,)
- `stl10_dino_test_embeddings.pt`  — dict with keys `features` (8000, 384) and `labels` (8000,)

These file names match exactly what `run_pipeline.py` and `test_pipeline.py` expect.
Once downloaded, place them at:
```
embeddings/stl10/stl10_dino_train_embeddings.pt
embeddings/stl10/stl10_dino_test_embeddings.pt
```

In [ ]:
# Mount Google Drive so we can save large files persistently
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE = 128
ROOT       = '/content'                      # STL-10 downloads here
SAVE_DIR   = '/content/drive/MyDrive/FYP'   # change if your Drive folder is different

os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Device : {DEVICE}')
print(f'Saving to: {SAVE_DIR}')

In [ ]:
# Standard DINO normalisation — must match what the pipeline uses
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
# Load DINO ViT-S/8 — frozen, eval mode
print('Loading DINO ViT-S/8...')
model = torch.hub.load('facebookresearch/dino:main', 'dino_vits8')
model.to(DEVICE)
model.eval()
for param in model.parameters():
    param.requires_grad = False
print('DINO loaded and frozen.')

In [ ]:
def extract_embeddings(split: str) -> dict:
    """
    Extracts L2-normalised DINO embeddings for one STL-10 split.

    Args:
        split: 'train' or 'test'

    Returns:
        dict with keys:
            'features': torch.Tensor (N, 384) — L2-normalised DINO embeddings
            'labels':   torch.Tensor (N,)     — ground truth class indices
    """
    print(f'\nExtracting {split} embeddings...')

    dataset = datasets.STL10(
        root=ROOT,
        split=split,
        download=True,
        transform=transform
    )

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,    # must be False — order must match label array
        num_workers=4,    # Colab can handle multiple workers, unlike Windows
        pin_memory=True
    )

    all_features = []
    all_labels   = []

    for batch_idx, (images, labels) in enumerate(loader):
        images = images.to(DEVICE)

        with torch.no_grad():
            features = model(images)                        # (B, 384)
        features = F.normalize(features, p=2, dim=1)       # L2 normalise

        all_features.append(features.cpu())
        all_labels.append(labels.cpu())

        if (batch_idx + 1) % 10 == 0:
            processed = (batch_idx + 1) * BATCH_SIZE
            print(f'  {processed} / {len(dataset)} images processed...')

    features = torch.cat(all_features, dim=0)   # (N, 384)
    labels   = torch.cat(all_labels,   dim=0)   # (N,)

    print(f'Done. features={features.shape}, labels={labels.shape}')
    return {'features': features, 'labels': labels}

In [ ]:
# Extract train split (5000 images)
train_data = extract_embeddings('train')

train_path = os.path.join(SAVE_DIR, 'stl10_dino_train_embeddings.pt')
torch.save(train_data, train_path)
print(f'Saved: {train_path}')

In [ ]:
# Extract test split (8000 images)
test_data = extract_embeddings('test')

test_path = os.path.join(SAVE_DIR, 'stl10_dino_test_embeddings.pt')
torch.save(test_data, test_path)
print(f'Saved: {test_path}')

In [ ]:
# Verification — confirm shapes and that no NaN values crept in
for name, path in [
    ('train', train_path),
    ('test',  test_path)
]:
    data = torch.load(path, weights_only=True)
    f, l = data['features'], data['labels']
    norms = f.norm(dim=1)
    print(f'{name}: features={f.shape}, labels={l.shape}, '
          f'norm_mean={norms.mean():.4f} (should be ~1.0), '
          f'has_nan={f.isnan().any().item()}')

## Next steps

Download both `.pt` files from your Drive and place them at:
```
Image-clustering-FYP/src/embeddings/stl10/stl10_dino_train_embeddings.pt
Image-clustering-FYP/src/embeddings/stl10/stl10_dino_test_embeddings.pt
```

Then run `python run_pipeline.py` locally. It will detect the saved embeddings
and skip extraction entirely, jumping straight to UMAP.